# E003 — Hyperparameter search for DCN-v2 and LightGBM

**Hypotheses:** none (supporting experiment) ([docs/hypotheses.md](../../docs/hypotheses.md))

**Question:** Which configurations to carry into the model comparison, chosen without touching the validation day?

**Method:** Optuna TPE, 60 DCN-v2 and 40 LightGBM trials, trained on 10-21..10-26 and selected on 10-27 NE. The validation day (10-28) stays clean for early stopping, calibration and comparisons.

In [1]:
from charade.analysis.experiment import setup

ctx = setup()
f"mode: {'smoke (fixture, tiny model)' if ctx.smoke else 'full data'}"

'mode: full data'

The full search takes ~16 minutes on the GTX 1660 SUPER and GPU training is not bit-reproducible, so in
full mode this notebook shows the committed search results; set `CHARADE_RERUN_TUNING=1` to rerun it.

In [2]:
import os

import polars as pl

from charade.models import tune

out = ctx.reports / "tuning"
if ctx.smoke:
    tune.run(trials_dcn=2, trials_gbdt=2, data_dir=ctx.data_dir, out=out)
elif os.environ.get("CHARADE_RERUN_TUNING") == "1":
    tune.run(out=out)
dcn, gbdt = pl.read_csv(out / "dcn.csv"), pl.read_csv(out / "gbdt.csv")
dcn.head(5)

/home/lucas/Developer/personal/take_home_assigment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


trial,value,hidden,embedding_dim,cross_layers,cross_rank,dropout,lr,weight_decay,batch_size,character_id_dropout,best_step
i64,f64,str,i64,i64,i64,f64,f64,f64,i64,f64,i64
34,0.874582,"""256-128""",24,2,64,0.299255,0.002934,0.000013,2048,0.081288,312
39,0.874804,"""512-256""",24,3,32,0.181916,0.00238,3.5316e-7,2048,0.013046,312
55,0.874813,"""512-256""",24,2,32,0.191617,0.002444,0.000006,2048,0.020202,312
49,0.874841,"""512-256""",24,3,32,0.200711,0.002519,0.000004,2048,0.097183,312
58,0.874862,"""512-256""",24,3,32,0.172891,0.002511,0.000002,2048,0.136569,312


In [3]:
gbdt.head(5)

trial,value,learning_rate,num_leaves,min_data_in_leaf,cat_smooth,cat_l2,feature_fraction,lambda_l2,best_iteration
i64,f64,f64,i64,i64,f64,f64,f64,f64,i64
51,0.874069,0.014444,254,323,26.853529,1.386834,0.404389,8.620825,479
55,0.87431,0.01392,164,405,26.579569,1.043862,0.400464,5.409424,470
58,0.874399,0.014172,152,297,26.067935,1.221364,0.440609,5.850657,612
54,0.874765,0.012902,235,168,20.392175,2.510468,0.433887,11.734412,476
53,0.874789,0.012274,197,212,14.762217,2.220572,0.425735,7.018004,469


In [4]:
shipped = ctx.settings.model
{"dcn": shipped.dcn.model_dump(), "gbdt": shipped.gbdt.model_dump()}

{'dcn': {'embedding_dim': 24,
  'cross_layers': 2,
  'cross_rank': 64,
  'hidden': [256, 128],
  'dropout': 0.3,
  'lr': 0.0029,
  'weight_decay': 1.3e-05,
  'batch_size': 2048,
  'max_epochs': 6,
  'evals_per_epoch': 4,
  'patience': 4,
  'character_id_dropout': 0.08,
  'seeds': [0, 1, 2]},
 'gbdt': {'learning_rate': 0.0144,
  'num_leaves': 254,
  'min_data_in_leaf': 323,
  'cat_smooth': 26.85,
  'cat_l2': 1.39,
  'feature_fraction': 0.4,
  'lambda_l2': 8.62,
  'max_rounds': 3000}}

## Reading
- The top DCN trials agree on 24-dim embeddings, batch 2048 and lr ~2.5-2.9e-3; architecture choices
  beyond that move inner NE by < 0.0003.
- With equal search effort (60 trials each), the best LightGBM trial (0.8741) matches the best DCN
  trial (0.8746) on the inner split: single tuned models are tied. An earlier 40-trial search had
  left LightGBM 0.004 behind, which overstated DCN's advantage.

## Conclusion
Equal effort (60 trials each). Best inner-split NE: DCN-v2 0.8746 (24-dim embeddings, 2 cross layers, 256-128 MLP, dropout 0.3, lr 2.9e-3, batch 2048) vs LightGBM 0.8741 (254 leaves, lr 0.014, feature fraction 0.40). Single tuned models are tied; both configs ship in [tool.charade.model].

**Decision:** Copy the best trials into `[tool.charade.model]` (done by hand, citing the CSVs).